In [91]:
import pandas as pd

In [110]:
 # Parse the Date and start_time columns as datetime

df_sleep = pd.read_csv("sleep_cleaned.csv") #, parse_dates=['Date', 'start_time', 'end_time'])
df_sleep['start_time'] = pd.to_datetime(df_sleep['start_time'], format="%H:%M:%S").dt.strftime("%H:%M:%S")
df_sleep['end_time'] = pd.to_datetime(df_sleep['end_time'], format="%H:%M:%S").dt.strftime("%H:%M:%S")

df_oxygen = pd.read_csv("oxygen_cleaned.csv") #, parse_dates=['Date', 'start_time'])
df_oxygen['sampling_time'] = pd.to_datetime(df_oxygen['sampling_time'], format="%H:%M:%S").dt.strftime("%H:%M:%S")

df_sleep.shape, df_oxygen.shape


((100, 5), (100, 3))

In [93]:
print("In sleep :", set(df_sleep['Date'].unique()))

print("In oxygen:", set(df_oxygen['Date'].unique()))

print("Difference:", set(df_oxygen['Date'].unique()).symmetric_difference(set(df_sleep['Date'].unique())))

In sleep : {'2024-09-28', '2024-09-21', '2024-10-01', '2024-09-20', '2024-09-25', '2024-09-29'}
In oxygen: {'2024-09-26', '2024-09-27', '2024-09-28', '2024-10-01', '2024-09-24', '2024-09-25', '2024-09-29', '2024-09-30'}
Difference: {'2024-09-26', '2024-09-27', '2024-09-21', '2024-09-24', '2024-09-20', '2024-09-30'}


In [119]:
# Full outer join on the 'Date' column
df_join = pd.merge(df_sleep, df_oxygen, on='Date', how='left', suffixes=('_sleep', '_oxygen'))

print(df_join.shape)

print(df_join[df_join['spo2_(%)'].isna()].shape)

# Replace NaN values in 'spo2_(%)' with 95
df_join.loc[df_join['spo2_(%)'].isna(), 'spo2_(%)'] = 95

# Replace NaN values in 'sampling_time' with the corresponding 'start_time' values
df_join.loc[df_join['spo2_(%)'] == 95, 'sampling_time'] = df_join['start_time']

print(df_join[df_join['spo2_(%)'].isna()].shape)


(1444, 7)
(22, 7)
(0, 7)


In [120]:

df_join['oxygen_within_sleep_interval'] = (
    (df_join['sampling_time'] >= df_join['start_time']) & 
    (df_join['sampling_time'] <= df_join['end_time'])
)

df_join = df_join[df_join['oxygen_within_sleep_interval'] == True]

print(df_join.shape)

df_join_filtered = df_join.groupby(by=['cycle_type','Date', 'start_time', 'end_time', 'duration_(s)']).agg(
    average_spo2=('spo2_(%)', 'mean'),
    spo2_samples=('spo2_(%)', 'count') 
).reset_index()


df_join_filtered


(126, 8)


,cycle_type,Date,start_time,end_time,duration_(s),average_spo2,spo2_samples
0,1,2024-09-21,01:08:00,01:10:00,120.0,95.0,1
1,1,2024-09-21,01:13:00,01:19:30,390.0,95.0,1
2,1,2024-09-21,01:21:00,02:15:30,3270.0,95.0,1
3,1,2024-09-21,02:16:00,02:17:00,60.0,95.0,1
4,1,2024-09-29,07:40:00,08:39:30,3570.0,95.5,4
...,...,...,...,...,...,...,...
75,5,2024-10-01,04:21:00,04:22:00,60.0,95.0,1
76,5,2024-10-01,04:32:00,04:39:30,450.0,97.5,2
77,5,2024-10-01,04:44:00,05:03:30,1170.0,95.0,1
78,5,2024-10-01,06:12:00,06:57:00,2700.0,97.5,2
